# Wykład 8 — Od danych do wyników: pliki, algorytmy i eksperymenty

Masz serię pomiarów. Dziś chcesz policzyć średnią, jutro posortować te same dane, a koleżanka chce użyć ich w swoim programie. Jeśli za każdym razem ktoś ma je przepisywać, wcześniej czy później pojawi się dodatkowy pomiar. Zwykle nieplanowany.

Po tym wykładzie uzgodnisz format pliku między programami, bezpiecznie odczytasz dane, wybierzesz sposób ich przetwarzania i zapiszesz wynik osobno. Zobaczysz dwa przepływy: pomiary i punkty.

Otwórz notebook w kernelu **C** i uruchamiaj komórki z katalogu `8/`. Ścieżka `w8_dane/pomiary.txt` oznacza plik w podkatalogu względem katalogu pracy kernela. Każda komórka C jest osobnym, kompletnym programem. Programy wymieniają dane przez pliki. Przy ćwiczeniach zostaw pliki wzorcowe i `samples/` w spokoju, a do własnych prób używaj nazw `w8_demo_*.txt`.

## W8.1 — Po co nam pliki?

Z W7 znamy `FILE *`, tablice i pętle. Tym razem **generator** i **analizator** mają osobne funkcje `main`. Generator może zakończyć pracę, a plik zostaje.

```text
generator → plik wejściowy → analizator → wynik na ekranie
                           → sortowanie → osobny plik
```

`fopen` zwraca uchwyt `FILE *` lub `NULL` przy błędzie. `r` służy do odczytu istniejącego pliku, `w` tworzy plik lub usuwa jego dotychczasową zawartość, a `a` dopisuje na końcu. Tryby `r+`, `w+` i `a+` pozwalają też na aktualizację, przy której obowiązują dodatkowe zasady przełączania między czytaniem i zapisem. Tutaj każdy strumień ma jeden kierunek.

**Przewidź.** Co stanie się z poprzednim wynikiem, jeśli ponownie otworzysz jego plik w trybie `w`? Kto potrzebuje trybu `r` w diagramie? `fclose` zamyka strumień i może zgłosić błąd, także gdy zapis czekał w buforze.

## W8.2 — Uzgadniamy format

Oba formaty zaczynają się od liczby rekordów $n$. Potem jest dokładnie $n$ pomiarów albo $n$ par. Nagłówek mówi, ile razy program ma czytać. Nie należy do danych.

| Plik | Nagłówek | Rekord |
|---|---|---|
| [pomiary.txt](w8_dane/pomiary.txt) | $1 \leq n \leq 200$ | jedna liczba całkowita od −99 do 99 |
| [punkty.txt](w8_dane/punkty.txt) | $1 \leq n \leq 200$ | dwie współrzędne $x\;y$ od −10 do 10 |

Zapisujemy nagłówek w pierwszym wierszu i po jednym rekordzie w kolejnych wierszach. Czytniki dopuszczają białe znaki jako separatory, więc jednakowo traktują spację i przejście do nowego wiersza. Nie dopuszczają dodatkowych pól. Współrzędne zapisujemy dziesiętnie, z kropką i bez notacji wykładniczej (maksymalnie 24 znaki na pole). W przykładach jedno pole całkowite ma najwyżej cztery znaki, wliczając ewentualny znak liczby.

**Przewidź.** Plik zawiera kolejno pola `3 10 20 30`. Które liczby są pomiarami? Co się stanie po zmianie nagłówka na `4`?

## W8.3 — Generator pomiarów

Do testowania analizatora przyda się plik, który da się odtworzyć. Przypomnij sobie `rand` i `srand` z W7. Tu ustawiamy ziarno raz, przed pętlą. `rand() % 199` daje resztę od 0 do 198, a odjęcie 99 przesuwa przedział na [−99, 99]. Reszty nie muszą mieć idealnie równych szans.

**Przewidź.** Ile wierszy zapisze generator dla `n = 6`? Co zobaczysz po ponownym uruchomieniu z tym samym ziarnem?

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    const int n = 6;
    const char *path = "w8_demo_pomiary.txt";
    if (n < 1 || n > 200) {
        fprintf(stderr, "Liczba pomiarów musi należeć do 1..200.\n");
        return 1;
    }

    FILE *file = fopen(path, "w");
    if (file == NULL) {
        perror(path);
        return 1;
    }

    srand(42);
    int failed = fprintf(file, "%d\n", n) < 0;
    for (int i = 0; !failed && i < n; ++i) {
        int value = rand() % 199 - 99;
        failed = fprintf(file, "%d\n", value) < 0;
    }
    if (fclose(file) != 0) failed = 1;
    if (failed) {
        fprintf(stderr, "Błąd zapisu lub zamknięcia pliku %s.\n", path);
        return 1;
    }

    printf("Zapisano %d pomiarów do %s.\n", n, path);
    return 0;
}

Otwórz `w8_demo_pomiary.txt`. Pierwszy wiersz zawiera 6, a po nim jest sześć wartości. Stałe ziarno daje tę samą sekwencję w danym środowisku. Inna biblioteka C może zwrócić inną sekwencję.

**Zmień** `n` na 8 i sprawdź osobno nagłówek, liczbę wierszy danych i zakres pomiarów. Która instrukcja odpowiada za każdą z tych trzech własności? Na koniec przywróć `n = 6`.

## W8.4 — Analiza pomiarów

Prowadzimy dwie rachuby. **Pojemność** tablicy to 200, a liczba aktywnych elementów to odczytane $n$. Po sprawdzeniu nagłówka wczytujemy wartości do tablicy. Pierwszy przebieg oblicza sumę i średnią $\bar{x}=\frac{1}{n}\sum_{i=0}^{n-1}x_i$. Drugi porównuje każdą wartość z już znaną średnią. Przy $n=0$ ten rachunek nie miałby sensu.

**Przewidź.** W danych wzorcowych jest sześć pomiarów: 3, 9, 5, −1, 4, 4. Suma to 24 i średnia to 4. Które pomiary trafią do każdej z trzech kategorii?

Czytnik przenosi pomysł z W7. `%4d` ogranicza konwersję do czterech znaków, więc wynik mieści się w `int`. Bezpośrednio następujące `%c` kontroluje granicę pola. `9x` i `12345` zostaną odrzucone, chociaż sam początek każdego pola dałoby się zamienić na liczbę. Koniec pliku zaraz po pełnej liczbie też jest poprawną granicą.

In [ ]:
#include <ctype.h>
#include <stdio.h>

#define MAX_SIZE 200

/* 1: całe pole, 0: zły format, EOF: brak pola lub błąd strumienia. */
int read_int(FILE *file, int *value)
{
    char separator;
    int status = fscanf(file, "%4d%c", value, &separator);
    if (ferror(file)) return EOF;
    if (status == 2 && isspace((unsigned char)separator)) return 1;
    if (status == 1 && feof(file)) return 1;
    return status == EOF ? EOF : 0;
}

void report_field(FILE *file, int status, const char *field)
{
    if (ferror(file)) perror("Odczyt");
    else if (status == EOF) fprintf(stderr, "%s: przedwczesny koniec pliku.\n", field);
    else fprintf(stderr, "%s: zły format lub wartość poza zakresem.\n", field);
}

int load_measurements(const char *path, int values[], int *count)
{
    FILE *file = fopen(path, "r");
    if (file == NULL) {
        perror(path);
        return 1;
    }

    int n;
    int status = read_int(file, &n);
    int failed = status != 1 || n < 1 || n > MAX_SIZE;
    if (failed) report_field(file, status, "Nagłówek");

    for (int i = 0; !failed && i < n; ++i) {
        status = read_int(file, &values[i]);
        if (status != 1 || values[i] < -99 || values[i] > 99) {
            report_field(file, status, "Pomiar");
            failed = 1;
        }
    }

    if (!failed) {
        char extra;
        if (fscanf(file, " %c", &extra) != EOF) {
            fprintf(stderr, "Dodatkowe dane po ostatnim pomiarze.\n");
            failed = 1;
        } else if (ferror(file)) {
            perror("Odczyt");
            failed = 1;
        }
    }
    if (fclose(file) != 0) {
        perror("Zamknięcie wejścia");
        failed = 1;
    }
    if (!failed) *count = n;
    return failed;
}

int main(void)
{
    int values[MAX_SIZE];
    int n;
    if (load_measurements("w8_dane/pomiary.txt", values, &n)) return 1;

    double sum = 0.0;
    for (int i = 0; i < n; ++i) sum += values[i];
    double mean = sum / n;

    int below = 0, equal = 0, above = 0;
    for (int i = 0; i < n; ++i) {
        if (values[i] < mean) ++below;
        else if (values[i] > mean) ++above;
        else ++equal;
    }

    printf("n=%d, suma=%.0f, średnia=%.2f\n", n, sum, mean);
    printf("poniżej=%d, równe=%d, powyżej=%d, razem=%d\n",
           below, equal, above, below + equal + above);
    return 0;
}

Każda z trzech kategorii ma po dwa elementy. Ich liczności muszą dodać się do $n$. Średnia na ekranie jest zaokrąglona, ale porównania wykonano z wartością przechowywaną w `mean`.

**Zmień** ścieżkę wejścia na `w8_demo_pomiary.txt` po uruchomieniu generatora z W8.3. Sprawdź sumę liczności. Dlaczego sama bieżąca suma, bez zachowania wartości, nie wystarczy do zliczenia kategorii po jednym odczycie pliku?

## W8.5 — Sortowanie i zapis

Plik wynikowy ma ten sam format co wejście, więc można go ponownie podać analizatorowi. Użyjemy sortowania bąbelkowego z W7. Po poprawnym odczycie zamieniamy sąsiadów w tablicy, a dopiero później otwieramy osobny plik wynikowy.

**Przewidź.** W jakiej kolejności zostaną zapisane dane wzorcowe? Co stanie się z dwiema czwórkami?

In [ ]:
#include <ctype.h>
#include <stdio.h>

#define MAX_SIZE 200

/* 1: całe pole, 0: zły format, EOF: brak pola lub błąd strumienia. */
int read_int(FILE *file, int *value)
{
    char separator;
    int status = fscanf(file, "%4d%c", value, &separator);
    if (ferror(file)) return EOF;
    if (status == 2 && isspace((unsigned char)separator)) return 1;
    if (status == 1 && feof(file)) return 1;
    return status == EOF ? EOF : 0;
}

void report_field(FILE *file, int status, const char *field)
{
    if (ferror(file)) perror("Odczyt");
    else if (status == EOF) fprintf(stderr, "%s: przedwczesny koniec pliku.\n", field);
    else fprintf(stderr, "%s: zły format lub wartość poza zakresem.\n", field);
}

int load_measurements(const char *path, int values[], int *count)
{
    FILE *file = fopen(path, "r");
    if (file == NULL) {
        perror(path);
        return 1;
    }

    int n;
    int status = read_int(file, &n);
    int failed = status != 1 || n < 1 || n > MAX_SIZE;
    if (failed) report_field(file, status, "Nagłówek");

    for (int i = 0; !failed && i < n; ++i) {
        status = read_int(file, &values[i]);
        if (status != 1 || values[i] < -99 || values[i] > 99) {
            report_field(file, status, "Pomiar");
            failed = 1;
        }
    }

    if (!failed) {
        char extra;
        if (fscanf(file, " %c", &extra) != EOF) {
            fprintf(stderr, "Dodatkowe dane po ostatnim pomiarze.\n");
            failed = 1;
        } else if (ferror(file)) {
            perror("Odczyt");
            failed = 1;
        }
    }
    if (fclose(file) != 0) {
        perror("Zamknięcie wejścia");
        failed = 1;
    }
    if (!failed) *count = n;
    return failed;
}

void bubble_sort(int values[], int n)
{
    for (int end = n - 1; end > 0; --end) {
        int swapped = 0;
        for (int i = 0; i < end; ++i) {
            if (values[i] > values[i + 1]) {
                int tmp = values[i];
                values[i] = values[i + 1];
                values[i + 1] = tmp;
                swapped = 1;
            }
        }
        if (!swapped) break;
    }
}

int save_measurements(const char *path, const int values[], int n)
{
    FILE *file = fopen(path, "w");
    if (file == NULL) {
        perror(path);
        return 1;
    }

    int failed = fprintf(file, "%d\n", n) < 0;
    for (int i = 0; !failed && i < n; ++i) {
        failed = fprintf(file, "%d\n", values[i]) < 0;
    }
    if (fclose(file) != 0) failed = 1;
    if (failed) fprintf(stderr, "Błąd zapisu lub zamknięcia pliku %s.\n", path);
    return failed;
}

int main(void)
{
    int values[MAX_SIZE];
    int n;
    const char *result = "w8_demo_posortowane.txt";
    if (load_measurements("w8_dane/pomiary.txt", values, &n)) return 1;

    bubble_sort(values, n);
    if (save_measurements(result, values, n)) return 1;
    printf("Zapisano %d posortowanych pomiarów do %s.\n", n, result);
    return 0;
}

Wynik po nagłówku to −1, 3, 4, 4, 5, 9. Sortowanie zmienia kolejność, zachowując liczbę rekordów i duplikaty. Otwórz oba pliki i sprawdź, gdzie zmieniła się kolejność.

**Zmień** wejście analizatora na `w8_demo_posortowane.txt`. Które statystyki pozostaną takie same? Przewidź też, gdzie zakończy się pętla sortowania po podaniu jej już uporządkowanych danych.

## W8.6 — Od funkcji do punktów

Chcemy narysować $y=\sin(x)$ dla $x$ od 0 do $2\pi$. Mając $n$ punktów, mamy $n-1$ odstępów. Liczymy każdy punkt z całkowitego indeksu $i$:

$$
x_i = x_{\min} + i\frac{x_{\max}-x_{\min}}{n-1}, \qquad i=0,\ldots,n-1.
$$

Dla $n=1$ wzór dzieliłby przez zero. Pierwszy i ostatni indeks odpowiadają końcom przedziału, z dokładnością arytmetyki zmiennoprzecinkowej. `acos(-1.0)` daje przybliżenie $\pi$ z `<math.h>`. Kod korzysta z biblioteki matematycznej. Opcja linkera to `-lm`, a pierwsza linia komórki przekazuje ją kernelowi C.

**Przewidź.** Dla $n=9$ wskaż indeksy punktów przy $0,\pi/2,\pi,3\pi/2,2\pi$ i podaj w przybliżeniu ich wartości $y$.

In [ ]:
//%ldflags:-lm
#include <math.h>
#include <stdio.h>

int main(void)
{
    const int n = 9;
    const char *path = "w8_demo_sinus.txt";
    if (n < 2 || n > 200) {
        fprintf(stderr, "Liczba próbek sinusa musi należeć do 2..200.\n");
        return 1;
    }

    FILE *file = fopen(path, "w");
    if (file == NULL) {
        perror(path);
        return 1;
    }

    const double pi = acos(-1.0);
    const double x_min = 0.0, x_max = 2.0 * pi;
    const double h = (x_max - x_min) / (n - 1);
    int failed = fprintf(file, "%d\n", n) < 0;
    for (int i = 0; !failed && i < n; ++i) {
        double x = x_min + i * h;
        failed = fprintf(file, "%.12f %.12f\n", x, sin(x)) < 0;
    }
    if (fclose(file) != 0) failed = 1;
    if (failed) {
        fprintf(stderr, "Błąd zapisu lub zamknięcia pliku %s.\n", path);
        return 1;
    }

    printf("Zapisano %d punktów do %s.\n", n, path);
    return 0;
}

Indeksy 0, 2, 4, 6 i 8 dają wartości bliskie 0, 1, 0, −1, 0. W pliku jest nagłówek i dziewięć par. Zapis ma 12 miejsc po kropce, więc mała wartość w pobliżu zera może wyglądać jak `-0.000000000000`. Porównując zapisane współrzędne z wartościami oczekiwanymi dla tego przykładu, przyjmij tolerancję $10^{-11}$.

**Zmień** `n` na 17. Ile razy zmniejszył się krok? Użyj par do narysowania wykresu w znanym Ci narzędziu. Pomiń nagłówek, bo opisuje liczbę punktów.

## W8.7 — Czy punkt należy do koła?

Tym razem pytanie dotyczy każdego punktu z osobna: czy należy do koła o środku $(0,0)$ i promieniu 1? Porównujemy $d^2=x^2+y^2$ z 1. Warunek `d2 <= 1.0` obejmuje wnętrze i brzeg. Liczymy je osobno, a po odczycie pary potrzebujemy już tylko trzech liczników.

**Przewidź.** Sklasyfikuj $(0,0)$, $(1,0)$, $(0,-1)$, $(2,0)$, $(-0{,}5,0{,}5)$ i $(-1,-1)$. Czy zmiana znaku jednej współrzędnej zmieni wynik?

W przygotowanym pliku po nagłówku są pary rozdzielone białymi znakami. `fscanf(file, "%lf %lf", &x, &y)` zwróciłoby 2 po dwóch udanych konwersjach. Ten program sprawdza najpierw ograniczoną postać tekstową **każdego pola**, a następnie przelicza je przez `strtod`. Dzięki temu odrzuca też za długie pola, przecinki i zapis wykładniczy. `end` wskazuje miejsce, w którym skończyła się konwersja. Pełna para to dwa poprawne pola, również gdy dzieli je koniec wiersza.

In [ ]:
#include <ctype.h>
#include <stdio.h>
#include <stdlib.h>

/* 1: całe pole, 0: zły format, EOF: brak pola lub błąd strumienia. */
int read_int(FILE *file, int *value)
{
    char separator;
    int status = fscanf(file, "%4d%c", value, &separator);
    if (ferror(file)) return EOF;
    if (status == 2 && isspace((unsigned char)separator)) return 1;
    if (status == 1 && feof(file)) return 1;
    return status == EOF ? EOF : 0;
}

void report_field(FILE *file, int status, const char *field)
{
    if (ferror(file)) perror("Odczyt");
    else if (status == EOF) fprintf(stderr, "%s: przedwczesny koniec pliku.\n", field);
    else fprintf(stderr, "%s: zły format lub wartość poza zakresem.\n", field);
}

int read_coordinate(FILE *file, double *value)
{
    char token[25], separator;
    /* Do 24 znaków pola, potem separator lub koniec pliku. */
    int status = fscanf(file, " %24[0123456789+.-]%c", token, &separator);
    if (ferror(file) || status == EOF) return EOF;
    if (!(status == 2 && isspace((unsigned char)separator))
        && !(status == 1 && feof(file))) return 0;

    char *end;
    double number = strtod(token, &end);
    if (end == token || *end != '\0' || number < -10.0 || number > 10.0) return 0;
    *value = number;
    return 1;
}

int main(void)
{
    const char *path = "w8_dane/punkty.txt";
    FILE *file = fopen(path, "r");
    if (file == NULL) {
        perror(path);
        return 1;
    }

    int n;
    int status = read_int(file, &n);
    int failed = status != 1 || n < 1 || n > 200;
    if (failed) report_field(file, status, "Nagłówek");

    int inside = 0, boundary = 0, outside = 0;
    for (int i = 0; !failed && i < n; ++i) {
        double x, y;
        status = read_coordinate(file, &x);
        if (status == 1) status = read_coordinate(file, &y);
        if (status != 1) {
            fprintf(stderr, "Rekord %d. ", i + 1);
            report_field(file, status, "Współrzędna");
            failed = 1;
            break;
        }

        double d2 = x*x + y*y;
        if (d2 < 1.0) ++inside;
        else if (d2 == 1.0) ++boundary;
        else ++outside;
    }

    if (!failed) {
        char extra;
        if (fscanf(file, " %c", &extra) != EOF) {
            fprintf(stderr, "Dodatkowe dane po ostatniej parze.\n");
            failed = 1;
        } else if (ferror(file)) {
            perror("Odczyt");
            failed = 1;
        }
    }
    if (fclose(file) != 0) {
        perror("Zamknięcie wejścia");
        failed = 1;
    }
    if (failed) return 1;

    printf("n=%d, wnętrze=%d, brzeg=%d, zewnątrz=%d, razem=%d\n",
           n, inside, boundary, outside, inside + boundary + outside);
    return 0;
}

Wzorzec daje po dwa punkty w każdej kategorii. Punkty `(1,0)` i `(0,-1)` dają dokładnie `d2 == 1.0`. Dla innych danych stosujemy dokładne porównanie **liczby obliczonej przez program** z 1. Punkt matematycznie leżący na brzegu może, wskutek zaokrągleń, trafić do innej kategorii.

**Zmień** ścieżkę wejściową na `w8_demo_sinus.txt` po wykonaniu W8.6. Dlaczego wciąż możesz użyć tego samego czytnika? Czy punkt o $x>1$ może znaleźć się we wnętrzu koła?

## W8.8 — Co może pójść źle?

Program może nie dostać żadnego pliku, dostać niepełny zapis albo stracić możliwość zapisania wyniku. Inaczej zareagujesz na literę zamiast liczby, inaczej na koniec pliku w połowie danych, a inaczej na błąd strumienia.

Dla pojedynczego `%d` udane `fscanf` daje 1, a napotkanie pola, które nie pasuje, zwykle daje 0. Jeśli przed pierwszą konwersją zabraknie wejścia, funkcja zwraca `EOF`. Sprawdź wtedy `ferror`, aby odróżnić błąd od zwykłego końca. Nasze czytniki dodatkowo sprawdzają granicę pola i odrzucają dane po ostatnim oczekiwanym rekordzie. Uzgodniony format ma ograniczone pola, więc możemy poprzestać na tych kontrolach.

| Bezpieczna próba | Co powinien zgłosić program? |
|---|---|
| W analizatorze wskaż nieistniejący plik `w8_demo_brak.txt`. | Błąd otwarcia. |
| Własny plik `w8_demo_proba.txt` zawiera nagłówek `3` oraz tylko `7` i `8`. | Przedwczesny koniec pliku przy trzecim pomiarze. |
| Zamiast trzeciej liczby wpisz `kot`. | Zły format pomiaru. |
| Ustaw nagłówek na `0` albo `201`. | Nagłówek poza zakresem. |
| Wpisz kolejno `2`, `7`, `8`, `---`, `7`, `8`. | Dodatkowe dane. Doklejona lista łamie uzgodniony format, nawet jeśli jest posortowana. |
| W pliku punktów zapowiedz jedną parę i zapisz tylko `0.5`. | Brak drugiej współrzędnej. |
| W generatorze wskaż jako wynik `w8_dane` (istniejący katalog). | Błąd otwarcia pliku do zapisu. |

W próbach ze zmienionymi danymi wskaż własny plik w odpowiedniej komórce. Po próbie przywróć oryginalne ścieżki. Po błędzie wejścia sortowanie nie otwiera wyniku. Po błędzie zapisu może zostać niepełny plik, a po błędzie wcześniejszym może nadal istnieć stary wynik. Samo istnienie pliku nie świadczy o sukcesie. Sprawdź komunikat i kod zakończenia programu.

**Sprawdź.** Dlaczego kontrolujemy zarówno `fprintf`, jak i `fclose`? Zapis może zostać zbuforowany, a błąd ujawnić się dopiero przy zamykaniu.

## W8.9 — Integracja i kontrola

W przepływie pomiarowym generator zapisuje `w8_demo_pomiary.txt`. Ten plik może wczytać analizator lub sortowanie. Sortowanie zapisuje `w8_demo_posortowane.txt`, które również można podać analizatorowi. W przepływie punktowym generator zapisuje `w8_demo_sinus.txt`, a klasyfikator odczytuje go parami. Wszystkie czytniki da się najpierw wypróbować na gotowych danych z `w8_dane/`.

### Dwa doświadczenia do samodzielnego wykonania

1. **Pomiary.** Uruchom generator z innym ziarnem. Włącz jego plik jako wejście analizatora i sortowania, a w ostatniej próbie podaj analizatorowi plik już posortowany. Przed wykonaniem ostatniej próby przewidź, czy zmieni się średnia lub liczności. Poszukaj również danych, dla których wszystkie pomiary są równe średniej. Sprawdź, czy wynik potwierdza przewidywanie.
2. **Punkty.** Ustaw 17 próbek sinusa, narysuj wykres z par i użyj tych samych par do klasyfikacji. W osobnym pliku dodaj punkty `(-1,0)` i `(0,-0.5)`, aktualizując nagłówek. Przewidź ich kategorie. W jakim pytaniu przydałoby się porównanie odległości z innym promieniem?

### Pytania kontrolne

1. Co zmienia nagłówek i kiedy można zauważyć, że nie zgadza się z danymi?
2. Dlaczego średnią musimy znać przed zliczeniem jej trzech kategorii?
3. Dlaczego do klasyfikacji punktów wystarczy bieżąca para, a użyte sortowanie trzyma pomiary w tablicy?
4. Jaką własność wyniku sortowania sprawdzisz oprócz rosnącej kolejności?
5. Które punkty przestaniesz zaliczać do koła, jeśli „należy do koła” ograniczysz do samego wnętrza?
6. Czy `fscanf` w naszych przykładach wymaga jednego rekordu w każdym wierszu?

<details>
<summary>Odpowiedzi po własnej próbie</summary>

1. Nagłówek ustala liczbę rekordów. Za duży ujawni się przy braku kolejnych poprawnych pól, a za mały przy sprawdzaniu danych za ostatnim rekordem.
2. Każdą wartość porównujemy ze średnią wszystkich pomiarów. Wcześniejsza, częściowa średnia może dać inną kategorię.
3. Kategoria punktu zależy od jego własnych współrzędnych. Bubble sort porównuje i zamienia sąsiadów przez kolejne przebiegi po tej samej tablicy.
4. Wynik ma tę samą liczbę rekordów i tyle samo wystąpień każdej wartości. Musi dać się ponownie odczytać w tym samym formacie.
5. Punkty na brzegu przestaną być zaliczane do koła, na przykład `(1,0)` i `(0,-1)`.
6. Wymaga separatorów między polami. Biały znak może być spacją, tabulatorem lub końcem wiersza.

</details>

Tu tablica miała ustaloną pojemność, a parę współrzędnych trzymaliśmy w dwóch zmiennych. W następnym wykładzie przyjrzymy się danym z układem wierszy i kolumn oraz przypadkom, gdy rozmiar poznajemy dopiero podczas działania programu.